# Notebook 02: Quantum Walks and Decision-Making

**QCCCM** — Quantum Compute for Computational Cognitive Modeling

---

## What you'll learn

1. How **classical random walks** model evidence accumulation (drift-diffusion)
2. How **quantum walks** differ: ballistic spreading vs diffusive spreading
3. **First-passage time** density as a model of reaction times
4. Why quantum interference produces **non-classical RT distributions**
5. Fitting quantum walk parameters to behavioral data

### Prerequisites
- NB 01 (density matrices, entropy, Bloch sphere)
- Basic familiarity with random walks

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from qcccm.core import (
    QuantumWalkParams,
    quantum_walk_evolution,
    quantum_walk_fpt_density,
    classical_vs_quantum_spreading,
    hadamard_coin,
    biased_coin,
)
from qcccm.viz.walks import (
    plot_qw_probability_evolution,
    plot_spreading_comparison,
    plot_fpt_density,
)

plt.rcParams['figure.dpi'] = 120
print(f"JAX backend: {jax.default_backend()}")

## 1. Classical Random Walk as Evidence Accumulation

In cognitive neuroscience, decisions are modeled as **evidence accumulation**: noisy evidence drifts toward a decision boundary. The simplest version is a **symmetric random walk** on a 1-D lattice.

At each time step, the walker moves left or right with equal probability:
- Position: $X_t = X_{t-1} \pm 1$
- Variance: $\text{Var}(X_t) = t$ (diffusive spreading)

The **reaction time** is the **first-passage time** — when the walker first reaches a decision boundary. For classical random walks, this follows an inverse-Gaussian distribution (the Wald distribution), which is the basis of the **drift-diffusion model** (DDM) in psychology.

In [ ]:
# Classical random walk simulation
np.random.seed(42)
n_steps = 100
n_walkers = 500

positions = np.zeros((n_walkers, n_steps + 1))
for t in range(n_steps):
    steps = np.random.choice([-1, 1], size=n_walkers)
    positions[:, t + 1] = positions[:, t] + steps

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample trajectories
for i in range(20):
    axes[0].plot(positions[i], alpha=0.4, linewidth=0.8)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].set_xlabel('Time step')
axes[0].set_ylabel('Position')
axes[0].set_title('Classical Random Walk Trajectories')

# Variance growth
times = np.arange(n_steps + 1)
empirical_var = np.var(positions, axis=0)
axes[1].plot(times, empirical_var, 'b-', label='Empirical variance', linewidth=2)
axes[1].plot(times, times, 'r--', label='Var = t (theory)', linewidth=2)
axes[1].set_xlabel('Time step')
axes[1].set_ylabel('Variance')
axes[1].set_title('Diffusive Spreading: Var ∝ t')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Quantum Walk: Ballistic Spreading

A **quantum walk** replaces classical coin flips with a quantum coin (a unitary operator on a 2-D coin space). The walker exists in a **superposition** of positions, and interference between paths changes the spreading dynamics fundamentally:

- **Classical**: Var$(X_t) = t$ (diffusive, $\sigma \propto \sqrt{t}$)
- **Quantum**: Var$(X_t) \propto t^2$ (ballistic, $\sigma \propto t$)

The quantum walker spreads **quadratically faster** than the classical walker. This has profound implications for evidence accumulation: quantum-like cognitive processes could reach decision boundaries faster.

In [ ]:
# Quantum walk evolution
params = QuantumWalkParams(n_sites=101, n_steps=50, start_pos=50)
probs = quantum_walk_evolution(params)

# Plot probability evolution at selected time steps
fig = plot_qw_probability_evolution(probs, times=[0, 10, 20, 30, 50])
plt.show()

In [ ]:
# Compare classical vs quantum spreading
times, c_var, q_var = classical_vs_quantum_spreading(n_sites=201, n_steps=80)

fig = plot_spreading_comparison(times, c_var, q_var)
plt.show()

print(f"At t=80:")
print(f"  Classical variance: {float(c_var[-1]):.1f}")
print(f"  Quantum variance:   {float(q_var[-1]):.1f}")
print(f"  Quantum/Classical ratio: {float(q_var[-1]/c_var[-1]):.1f}x")

## 3. The Coin Operator: Tuning the Walk

The **coin angle** $\theta$ controls the walk's behavior:
- $\theta = \pi/4$: Hadamard coin (standard symmetric walk)
- $\theta < \pi/4$: Biased toward right
- $\theta > \pi/4$: Biased toward left

In the cognitive interpretation, $\theta$ represents the **evidence quality** — how strongly sensory input drives the walker toward one decision or the other.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, theta in enumerate([jnp.pi/6, jnp.pi/4, jnp.pi/3]):
    params = QuantumWalkParams(
        coin_angle=theta, n_sites=101, n_steps=40, start_pos=50
    )
    probs = quantum_walk_evolution(params)

    ax = axes[i]
    center = 50
    positions = np.arange(101) - center
    ax.plot(positions, np.asarray(probs[-1]), 'b-', linewidth=1.5)
    ax.fill_between(positions, np.asarray(probs[-1]), alpha=0.3)
    ax.set_xlabel('Position')
    ax.set_ylabel('Probability')
    ax.set_title(f'θ = {float(theta):.3f} ({float(theta/jnp.pi):.2f}π)')
    ax.set_xlim(-40, 40)

plt.suptitle('Effect of Coin Angle on Quantum Walk (t = 40)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. First-Passage Time: Quantum Reaction Times

The **first-passage time** (FPT) density is the probability that the walker first reaches an absorbing boundary at time $t$. In the decision-making analogy:

- **Absorbing boundary** = decision threshold
- **FPT density** = reaction time distribution

Classical FPT follows a smooth inverse-Gaussian. Quantum FPT shows **interference fringes** — oscillatory patterns arising from constructive and destructive interference between quantum paths.

In [ ]:
# FPT density: quantum walk hitting an absorbing boundary
fpt_params = QuantumWalkParams(
    n_sites=51, n_steps=80, start_pos=15, absorbing_right=35,
)
fpt_density = quantum_walk_fpt_density(fpt_params, boundary="right")

fig = plot_fpt_density(fpt_density, title="Quantum Walk First-Passage Time Density")
plt.show()

print(f"Total absorption probability: {float(jnp.sum(fpt_density)):.3f}")
print(f"Modal FPT (most likely RT): t = {int(jnp.argmax(fpt_density)) + 1}")
print(f"Mean FPT: t = {float(jnp.sum(jnp.arange(1, 81) * fpt_density) / jnp.sum(fpt_density)):.1f}")

In [ ]:
# Compare FPT for different coin angles (evidence strengths)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
thetas = [jnp.pi/6, jnp.pi/4, jnp.pi/3]
labels = ['Strong evidence', 'Balanced (Hadamard)', 'Weak evidence']

for i, (theta, label) in enumerate(zip(thetas, labels)):
    params = QuantumWalkParams(
        coin_angle=theta, n_sites=51, n_steps=80,
        start_pos=15, absorbing_right=35,
    )
    fpt = quantum_walk_fpt_density(params, boundary="right")
    plot_fpt_density(fpt, ax=axes[i], title=f'{label}\nθ = {float(theta):.3f}')

plt.suptitle('RT Distribution Depends on Evidence Quality (Coin Angle)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Quantum vs Classical FPT Comparison

The key prediction of quantum cognition for reaction times: **interference fringes**.

Classical DDM predicts smooth, unimodal RT distributions. Quantum walk FPT can show multi-modal structure from constructive/destructive interference. This is testable in behavioral experiments where subjects show oscillatory patterns in RT histograms.

In [ ]:
# Side-by-side: classical vs quantum FPT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quantum FPT
params = QuantumWalkParams(
    n_sites=51, n_steps=100, start_pos=15, absorbing_right=40,
)
q_fpt = quantum_walk_fpt_density(params, boundary="right")
t_q = np.arange(1, 101)
axes[0].bar(t_q, np.asarray(q_fpt), color='steelblue', alpha=0.7, width=0.8)
axes[0].set_xlabel('Time step (RT)')
axes[0].set_ylabel('FPT density')
axes[0].set_title('Quantum Walk FPT\n(note interference fringes)')

# Classical FPT (simulated)
np.random.seed(0)
classical_fpts = []
for _ in range(10000):
    pos = 15
    for t in range(1, 101):
        pos += np.random.choice([-1, 1])
        if pos >= 40:
            classical_fpts.append(t)
            break

axes[1].hist(classical_fpts, bins=np.arange(1, 101), density=True,
             color='coral', alpha=0.7)
axes[1].set_xlabel('Time step (RT)')
axes[1].set_ylabel('FPT density')
axes[1].set_title('Classical Random Walk FPT\n(smooth, no fringes)')

plt.suptitle('Quantum Interference Creates Non-Classical RT Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Exercises

### Basic

**Exercise 2.1:** Run the quantum walk with `n_sites=201, n_steps=100` and measure the variance at each time step. Fit a power law $\text{Var}(t) = c \cdot t^\alpha$ and verify that $\alpha \approx 2$ (ballistic) vs $\alpha \approx 1$ (diffusive).

**Exercise 2.2:** For a fixed absorbing boundary at position 30, vary the coin angle from $\pi/8$ to $3\pi/8$ in 10 steps. Plot the mean FPT as a function of $\theta$. What coin angle minimizes mean RT?

**Exercise 2.3:** Compare the FPT distribution for a quantum walk starting at position 10 vs position 20 (both with boundary at 35). How does starting distance affect the shape of the RT distribution?

### Stretch

**Exercise 2.4 (Zeno effect):** Add "measurement" by projecting the quantum state onto the computational basis (zeroing off-diagonal elements of the density matrix) every $k$ steps. Show that frequent measurement ($k=1$) recovers classical diffusive spreading, while infrequent measurement ($k \gg 1$) preserves ballistic spreading. This is the quantum Zeno effect applied to evidence accumulation.

**Exercise 2.5 (Biased evidence):** Modify the coin operator to create a biased walk with drift (modeling biased evidence). Show that the quantum walk reaches a decision boundary faster than the classical biased random walk for the same drift rate. Quantify the speedup.

## Next: NB 03

In Notebook 03, we compare **quantum vs classical Expected Free Energy** for active inference agents, showing how quantum interference changes policy selection.